In [ ]:
import numpy as np
import torch

ISTA

In [ ]:
def ISTA(Vandermonde, y, grad, lamb,  *args, epochs = 50, sched = False, lr = None, inference = False, cutoff = 0.5):
  x = np.random.random(size = Vandermonde.shape[1]).reshape(-1,1)

  if lr == None:
    lr = 1 / (np.linalg.norm(Vandermonde.T @ Vandermonde, ord = 2))

  soft_thres = lambda x, th = lamb * lr: np.sign(x) * np.maximum(np.abs(x) - th, 0)
  loss = []
  coef_loss = []
  for epoch in range(epochs):
    x -= lr * grad(Vandermonde, x, y)
    x = soft_thres(x)
    if inference:
        loss.append(((args[0] - Vandermonde @ x) ** 2).mean()) ## loss between ISTA predicted values and the true values
        coef_loss.append(((args[1] - x) ** 2).mean()) ## loss between ISTA coefficients and the true coefficients


  if inference:
      return loss, coef_loss, x

  ## performs cutoff and resulting least squares; we don't want this for inference
  active = np.abs(x) > cutoff

  x_debiased = np.zeros_like(x)
  x_debiased[active] = np.linalg.lstsq(
      Vandermonde[:, active.reshape(-1)],
      y,
      rcond=None
  )[0].reshape(-1)

  return x_debiased

FISTA

In [ ]:
## FISTA Algorithm

def FISTA(Vandermonde, y, epochs = 1000, lamb = 0.5, grad = lambda A, x, b: (A.T @ (A @ x - b)), cutoff = 0.5, lr = None): ## cutoff is the cutoff value for the least squares at the end
  x = torch.rand(size = (Vandermonde.shape[1],)).reshape(-1,1)
  if lr == None:
    lr = 1 / (torch.linalg.norm(Vandermonde.T @ Vandermonde, ord = 2)) ## lr guaranteeing convergence for fista

  Vandermonde = Vandermonde
  soft_thres = lambda x, th = lamb * lr: torch.sign(x) * torch.maximum(torch.abs(x) - th, torch.tensor(0)) ## soft thresholder function
  prev_step = torch.zeros(Vandermonde.shape[1]).reshape(-1,1) 

  tk = 1
  losses = []
  epochs_list = []
  for epoch in range(epochs):
    Beta = (tk - 1) / ((1 + sqrt(1 + 4 * tk ** 2)) / 2)
    prev_step = Beta * prev_step - lr * grad(Vandermonde, x + prev_step, y)
    x += prev_step
    x = soft_thres(x)

    tk = (1 + sqrt(1 + 4 * tk ** 2)) / 2
    
    losses.append((torch.linalg.vector_norm((A @ x - b), ord = 2).square() / (A @ x - b).size(0)).item()) ## MSE loss
    epochs_list.append(epoch)

  active = x.reshape(-1).abs() > cutoff

  x_debiased = np.zeros_like(x)

  x_debiased[active] = np.linalg.lstsq(
      Vandermonde[:, active.reshape(-1)],
      y,
      rcond=None
  )[0].reshape(-1,1)


  return x_debiased